In [1]:
import logging
import os

os.makedirs("../logs", exist_ok=True)

log_path = "../logs/validation.log"
logging.basicConfig(
    filename=log_path,
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

# Actividad 2.3 — Validación estructural y semántica de datos

## Objetivo
En este notebook trabajaremos la tercera etapa del pipeline de datos: la validación estructural y semántica.

A diferencia de las etapas anteriores, aquí no volveremos a realizar la ingesta ni la transformación, ya que ese proceso ya fue ejecutado previamente y el dataset resultante se encuentra en la carpeta `data/processed/`.

## Flujo del pipeline trabajado
1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica ← etapa actual

## Meta de esta actividad
- Leer el dataset procesado
- Aplicar validaciones estructurales
- Aplicar validaciones semánticas
- Separar registros válidos e inválidos
- Registrar observaciones en logs
- Exportar resultados para la siguiente etapa del pipeline

## Contexto de trabajo

En este proyecto ya existe una etapa previa de limpieza y transformación. Por lo tanto, este notebook asume que el dataset de entrada ya fue generado y almacenado en la carpeta:

`../data/processed/`

A partir de ese archivo, se verificará si los datos cumplen con reglas técnicas y lógicas antes de ser cargados posteriormente a una base de datos.

## Carga del dataset procesado

En esta etapa leeremos el archivo generado en la fase anterior del pipeline.

Si tu archivo tiene otro nombre, debes ajustar la ruta en la siguiente celda.

In [2]:
import pandas as pd
import os
import logging
import numpy as np

ruta_entrada = "../data/processed/fraude_clean.csv"
logging.info(f"Inicio de lectura del dataset procesado desde {ruta_entrada}")

df = pd.read_csv(ruta_entrada)
logging.info(f"Dataset procesado cargado correctamente: filas={len(df)}, columnas={len(df.columns)}")
df

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_year,edad
0,119106,NaN,377895991033232,"fraud_Bahringer, Schoen and Corkery",shopping_pos,1.07,Kimberly,Myers,F,6881 King Isle Suite 228,...,5438,"Librarian, academic",1964-11-17,cf581d75ccc9ba838a05dec8bfa78b5b,1375430128,41.240083,-71.837788,0,NaN,NaN
1,179292,NaN,30364087349027,"fraud_Romaguera, Wehner and Tromp",kids_pets,94.99,Samuel,Sandoval,M,0005 Morrison Land,...,7163,Fitness centre manager,1982-02-05,b1bfaf13224da41f422db483fd810dd7,1377266716,35.156537,-95.806648,0,NaN,NaN
2,540729,NaN,30328384440870,fraud_Berge-Hills,kids_pets,31.28,Helen,Campbell,F,182 Sergio Summit Apt. 129,...,602,Cytogeneticist,1954-07-14,cde9fc0136873645778d0ad8817db655,1388247749,39.888665,-93.106804,0,NaN,NaN
3,374360,NaN,30364087349027,"fraud_Connelly, Reichert and Fritsch",gas_transport,73.06,Samuel,Sandoval,M,0005 Morrison Land,...,7163,Fitness centre manager,1982-02-05,90b8429191e5c83df1afba4e5db4d61e,1384425890,36.734101,-96.737345,0,NaN,NaN
4,314574,NaN,4198470814557,fraud_Kuphal-Predovic,misc_net,9.99,Christie,Williamson,F,519 Jerry Views,...,2036,Engineering geologist,1971-08-20,e4893795b6b3e41667129b9ed13b9650,1382147409,40.922072,-94.899388,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,314327,2020-10-18 23:37:19,30290551782700,"fraud_Eichmann, Hayes and Treutel",travel,3.70,John,Clarke,M,27909 Peter Motorway,...,13422,Commissioning editor,1961-09-03,36ea9e7e39824b7a8e0081061ae67b36,1382139439,36.249354,-87.344734,0,2020.0,59.0
59996,180915,2020-08-23 23:39:29,4191109180926792,fraud_Kilback LLC,food_dining,126.90,Jennifer,Spencer,F,477 Wheeler Burg,...,4512,Farm manager,1992-10-03,d43ed88df408752c5ac66487a117e3c1,1377301169,39.048872,-82.792056,0,2020.0,28.0
59997,389411,2020-11-20 22:54:33,4997733566924489,fraud_Monahan-Morar,personal_care,14.06,Stephanie,Taylor,F,598 Martin Pine Suite 365,...,753116,Fisheries officer,1971-08-06,1e22a4979454ee9b29f85d038243675a,1384988073,45.316035,-93.082687,0,2020.0,49.0
59998,228131,2020-09-11 04:03:53,30033162392091,fraud_Predovic Inc,shopping_net,93.75,Kevin,Summers,M,7302 Samantha Mission,...,566,Metallurgist,1975-01-26,4bb2c4c7700eedf161ea43ef77a634f0,1378872233,41.993681,-96.840289,0,2020.0,45.0


## Exploración inicial

Antes de validar, revisaremos:
- estructura general del dataset
- tipos de datos
- posibles nulos remanentes

In [3]:
logging.info(f"Exploración inicial del dataset: filas={len(df)}, columnas={len(df.columns)}")
df.info()
logging.info("Se ejecutó df.info() para revisar estructura y tipos")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             60000 non-null  int64  
 1   trans_date_trans_time  59820 non-null  object 
 2   cc_num                 60000 non-null  int64  
 3   merchant               60000 non-null  object 
 4   category               60000 non-null  object 
 5   amt                    59820 non-null  float64
 6   first                  60000 non-null  object 
 7   last                   60000 non-null  object 
 8   gender                 60000 non-null  object 
 9   street                 60000 non-null  object 
 10  city                   60000 non-null  object 
 11  state                  60000 non-null  object 
 12  zip                    60000 non-null  int64  
 13  lat                    60000 non-null  float64
 14  long                   60000 non-null  float64
 15  ci

In [4]:
nulos_por_columna = df.isnull().sum()
logging.info(f"Conteo de nulos completado: nulos_totales={int(nulos_por_columna.sum())}")
nulos_por_columna

Unnamed: 0                 0
trans_date_trans_time    180
cc_num                     0
merchant                   0
category                   0
amt                      180
first                      0
last                       0
gender                     0
street                     0
city                       0
state                      0
zip                        0
lat                        0
long                       0
city_pop                   0
job                        0
dob                      180
trans_num                  0
unix_time                  0
merch_lat                  0
merch_long                 0
is_fraud                   0
trans_year               180
edad                     360
dtype: int64

## Configuración del log

Crearemos un archivo de log para dejar trazabilidad del proceso de validación.

In [5]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/validation.log",
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

logging.info("Inicio del proceso de validación del dataset procesado")
logging.info("Logger configurado para validación estructural y semántica")
print("Logger configurado correctamente")

Logger configurado correctamente


## Reglas de validación - Detección de Fraude

Aplicaremos dos tipos de validación específicas para detección de fraude:

### Validaciones estructurales
1. `trans_date_trans_time`: Fecha de transacción válida
2. `dob`: Fecha de nacimiento válida
3. `amt`: Monto debe ser numérico
4. `gender`: Género debe ser M o F
5. `is_fraud`: Debe ser 0 o 1

### Validaciones semánticas
1. El monto `amt` debe ser mayor a 0 (transacción válida)
2. La edad calculada desde `dob` debe estar entre 18 y 120 años
3. `lat` (latitud titular): Debe estar entre -90 y 90
4. `long` (longitud titular): Debe estar entre -180 y 180

## Aplicación de validaciones

Generaremos columnas booleanas para identificar si cada registro cumple o no con cada regla.

### Nota técnica: ¿Por qué validación vectorizada?

En lugar de usar funciones con `.apply()` (que procesa fila por fila), usamos **operaciones vectorizadas** de pandas/numpy que procesan la columna completa a nivel de C. Esto es **10-50x más rápido** con datasets grandes de fraude (cientos de miles de transacciones), mejorando significativamente el rendimiento sin perder claridad.

In [6]:
df_validacion = df.copy()
logging.info(f"Inicio de validaciones estructurales y semánticas: filas={len(df_validacion)}")

# Validaciones estructurales (VECTORIZADAS - MÁS RÁPIDO)
df_validacion["val_trans_date"] = pd.to_datetime(df_validacion["trans_date_trans_time"], errors='coerce').notna()
df_validacion["val_dob"] = pd.to_datetime(df_validacion["dob"], errors='coerce').notna()
df_validacion["val_amt"] = pd.to_numeric(df_validacion["amt"], errors='coerce').notna()
df_validacion["val_gender"] = df_validacion["gender"].str.upper().isin({"M", "F"})
df_validacion["val_is_fraud"] = df_validacion["is_fraud"].astype(str).isin({"0", "1"})

# Validaciones semánticas (VECTORIZADAS)
df_validacion["val_amt_semantica"] = pd.to_numeric(df_validacion["amt"], errors='coerce') > 0

# Edad vectorizada
dob = pd.to_datetime(df_validacion["dob"], errors='coerce')
trans_date = pd.to_datetime(df_validacion["trans_date_trans_time"], errors='coerce')
edad = (trans_date.dt.year - dob.dt.year) - ((trans_date.dt.month < dob.dt.month) | ((trans_date.dt.month == dob.dt.month) & (trans_date.dt.day < dob.dt.day)))
df_validacion["val_edad"] = (edad >= 18) & (edad <= 120)

# Rangos coordenadas (VECTORIZADOS)
df_validacion["val_lat_rango"] = (pd.to_numeric(df_validacion["lat"], errors='coerce').between(-90, 90))
df_validacion["val_long_rango"] = (pd.to_numeric(df_validacion["long"], errors='coerce').between(-180, 180))

val_trans_date_ok = int(df_validacion["val_trans_date"].sum())
val_amt_ok = int(df_validacion["val_amt"].sum())
val_gender_ok = int(df_validacion["val_gender"].sum())
logging.info(
    f"Validaciones calculadas: fecha_ok={val_trans_date_ok}, monto_ok={val_amt_ok}, genero_ok={val_gender_ok}"
)

df_validacion.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,edad,val_trans_date,val_dob,val_amt,val_gender,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango
0,119106,NaN,377895991033232,"fraud_Bahringer, Schoen and Corkery",shopping_pos,1.07,Kimberly,Myers,F,6881 King Isle Suite 228,...,NaN,False,True,True,True,True,True,False,True,True
1,179292,NaN,30364087349027,"fraud_Romaguera, Wehner and Tromp",kids_pets,94.99,Samuel,Sandoval,M,0005 Morrison Land,...,NaN,False,True,True,True,True,True,False,True,True
2,540729,NaN,30328384440870,fraud_Berge-Hills,kids_pets,31.28,Helen,Campbell,F,182 Sergio Summit Apt. 129,...,NaN,False,True,True,True,True,True,False,True,True
3,374360,NaN,30364087349027,"fraud_Connelly, Reichert and Fritsch",gas_transport,73.06,Samuel,Sandoval,M,0005 Morrison Land,...,NaN,False,True,True,True,True,True,False,True,True
4,314574,NaN,4198470814557,fraud_Kuphal-Predovic,misc_net,9.99,Christie,Williamson,F,519 Jerry Views,...,NaN,False,True,True,True,True,True,False,True,True


## Consolidación de la validación

Un registro será considerado válido solo si cumple todas las reglas de fraude definidas.

In [7]:
columnas_validacion = [
    "val_trans_date",
    "val_dob",
    "val_amt",
    "val_gender",
    "val_is_fraud",
    "val_amt_semantica",
    "val_edad",
    "val_lat_rango",
    "val_long_rango"
]

# Un registro es válido si pasa TODAS las validaciones
df_validacion["registro_valido"] = df_validacion[columnas_validacion].all(axis=1)

validos_totales = int(df_validacion["registro_valido"].sum())
logging.info(f"Consolidación completada: registros_validos={validos_totales}, registros_invalidos={len(df_validacion) - validos_totales}")

# Mostrar resumen de validación
df_validacion[["registro_valido"] + columnas_validacion].head()

,registro_valido,val_trans_date,val_dob,val_amt,val_gender,val_is_fraud,val_amt_semantica,val_edad,val_lat_rango,val_long_rango
0,False,False,True,True,True,True,True,False,True,True
1,False,False,True,True,True,True,True,False,True,True
2,False,False,True,True,True,True,True,False,True,True
3,False,False,True,True,True,True,True,False,True,True
4,False,False,True,True,True,True,True,False,True,True


## Generación de observaciones

Para cada registro inválido, documentamos qué validaciones fallaron.

In [8]:
def obtener_observaciones(row):
    errores = []

    # Validaciones estructurales
    if not row["val_trans_date"]:
        errores.append("Fecha transacción inválida")
    if not row["val_dob"]:
        errores.append("Fecha nacimiento inválida")
    if not row["val_amt"]:
        errores.append("Monto no numérico")
    if not row["val_gender"]:
        errores.append("Género no es M o F")
    if not row["val_is_fraud"]:
        errores.append("is_fraud no es 0 o 1")

    # Validaciones semánticas
    if row["val_amt"] and not row["val_amt_semantica"]:
        errores.append("Monto <= 0")
    if not row["val_edad"]:
        errores.append("Edad fuera de rango (18-120)")
    if not row["val_lat_rango"]:
        errores.append("Latitud fuera de rango (-90 a 90)")
    if not row["val_long_rango"]:
        errores.append("Longitud fuera de rango (-180 a 180)")

    return " | ".join(errores) if errores else "Registro válido"

# Generar observaciones para todos los registros
df_validacion["observaciones"] = df_validacion.apply(obtener_observaciones, axis=1)
invalidos_generados = int((df_validacion["observaciones"] != "Registro válido").sum())
logging.info(f"Observaciones generadas para el dataset: registros_con_error={invalidos_generados}")

# Mostrar registros con errores
df_validacion[["observaciones", "registro_valido"]].head(10)

,observaciones,registro_valido
0,Fecha transacción inválida | Edad fuera de ran...,False
1,Fecha transacción inválida | Edad fuera de ran...,False
2,Fecha transacción inválida | Edad fuera de ran...,False
3,Fecha transacción inválida | Edad fuera de ran...,False
4,Fecha transacción inválida | Edad fuera de ran...,False
5,Fecha transacción inválida | Edad fuera de ran...,False
6,Fecha transacción inválida | Edad fuera de ran...,False
7,Fecha transacción inválida | Edad fuera de ran...,False
8,Fecha transacción inválida | Edad fuera de ran...,False
9,Fecha transacción inválida | Edad fuera de ran...,False


## Separación de registros válidos e inválidos

In [9]:
# Separación de registros válidos e inválidos (caso FRAUDE)
df_validos = df_validacion[df_validacion["registro_valido"]].copy()
df_invalidos = df_validacion[~df_validacion["registro_valido"]].copy()

logging.info(
    f"Separación completada: total={len(df_validacion)}, validos={len(df_validos)}, invalidos={len(df_invalidos)}"
)

print("Cantidad de registros totales:", len(df_validacion))
print("Cantidad de registros válidos:", len(df_validos))
print("Cantidad de registros inválidos:", len(df_invalidos))

# Mostrar las observaciones más comunes en inválidos
if len(df_invalidos) > 0:
    print('\nTop motivos de invalidez:')
    top_motivos = df_invalidos['observaciones'].value_counts().head(10)
    logging.info("Top motivos de invalidez:\n" + top_motivos.to_string())
    print(top_motivos)
else:
    logging.info("No se encontraron registros inválidos en la validación")
    print('\nNo se encontraron registros inválidos.')

Cantidad de registros totales: 60000
Cantidad de registros válidos: 57513
Cantidad de registros inválidos: 2487

Top motivos de invalidez:
observaciones
Edad fuera de rango (18-120)                                 912
Longitud fuera de rango (-180 a 180)                         225
Monto <= 0                                                   223
Latitud fuera de rango (-90 a 90)                            223
Fecha transacción inválida | Edad fuera de rango (18-120)    180
is_fraud no es 0 o 1                                         180
Fecha nacimiento inválida | Edad fuera de rango (18-120)     180
Monto no numérico                                            179
Género no es M o F                                           174
Género no es M o F | Edad fuera de rango (18-120)              6
Name: count, dtype: int64


In [10]:
# Añadir metadatos y limpiar columnas relevantes para el caso de fraude

df_validos = df_validos.copy()
df_invalidos = df_invalidos.copy()

# Estado y origen
df_validos["estado_calidad"] = "valido"
df_invalidos["estado_calidad"] = "error"

df_validos["fuente"] = "python_notebook"
df_invalidos["fuente"] = "python_notebook"

# Copiar observaciones al campo estándar 'observacion'
df_invalidos["observacion"] = df_invalidos["observaciones"]

# Asegurar tipos numéricos relevantes: 'amt' y 'city_pop' cuando existan
if 'amt' in df_validos.columns:
    df_validos['amt'] = pd.to_numeric(df_validos['amt'], errors='coerce')
    df_invalidos['amt'] = pd.to_numeric(df_invalidos['amt'], errors='coerce')

if 'city_pop' in df_validos.columns:
    df_validos['city_pop'] = pd.to_numeric(df_validos['city_pop'], errors='coerce')
    df_invalidos['city_pop'] = pd.to_numeric(df_invalidos['city_pop'], errors='coerce')

logging.info(
    f"Metadatos asignados: validos={len(df_validos)}, invalidos={len(df_invalidos)}, columnas_clave=['estado_calidad','fuente','observacion']"
)

print('Ejemplos válidos:')
print(df_validos[['amt','registro_valido']].head())

print('\nEjemplos inválidos (observaciones):')
print(df_invalidos[['observaciones','registro_valido']].head())

Ejemplos válidos:
         amt  registro_valido
1800  103.63             True
1801    1.20             True
1802    5.84             True
1803    2.40             True
1804   47.36             True

Ejemplos inválidos (observaciones):
                                       observaciones  registro_valido
0  Fecha transacción inválida | Edad fuera de ran...            False
1  Fecha transacción inválida | Edad fuera de ran...            False
2  Fecha transacción inválida | Edad fuera de ran...            False
3  Fecha transacción inválida | Edad fuera de ran...            False
4  Fecha transacción inválida | Edad fuera de ran...            False


## Registro del proceso en logs

In [11]:
logging.info(f"Total registros procesados: {len(df_validacion)}")
logging.info(f"Total registros válidos: {len(df_validos)}")
logging.info(f"Total registros inválidos: {len(df_invalidos)}")

if len(df_invalidos) > 0:
    # Top motivos
    top_reasons = df_invalidos['observaciones'].value_counts().head(10)
    logging.info("Top motivos de invalidez:\n" + top_reasons.to_string())

    # Muestras de registros inválidos con columnas clave si existen
    sample_cols = ['trans_num', 'cc_num', 'amt', 'trans_date_trans_time', 'observaciones']
    sample_cols = [c for c in sample_cols if c in df_invalidos.columns]
    for idx, row in df_invalidos[sample_cols].head(10).iterrows():
        logging.warning(f"Registro inválido index={idx}: " + ", ".join([f"{c}={row[c]}" for c in sample_cols]))

logging.info('Registro de validación completado')
print("Información registrada en ../logs/validation.log")

Información registrada en ../logs/validation.log


## Exportación de resultados

Los archivos generados en esta etapa quedarán en la carpeta `processed`, ya que son resultados del pipeline listos para la siguiente fase.

In [12]:
os.makedirs("../data/processed", exist_ok=True)

valid_path = "../data/processed/fraude_validas.csv"
invalid_path = "../data/processed/fraude_invalidas.csv"

# Guardar sin índices
df_validos.to_csv(valid_path, index=False)
df_invalidos.to_csv(invalid_path, index=False)

logging.info(f"Exportación completada: validos={valid_path}, invalidos={invalid_path}")
print(f"Archivos exportados correctamente:\n - {valid_path}\n - {invalid_path}")

Archivos exportados correctamente:
 - ../data/processed/fraude_validas.csv
 - ../data/processed/fraude_invalidas.csv
